# ВАЖНО!!! решение сделано на GPU, поэтому разные запуски могут давать немного разные сабмиты, однако все, что получались у меня, всё равно обеспечивают топ 1 и на public, и на private


## 1. Импорты и настройки


In [ ]:
import os
from pathlib import Path
from functools import lru_cache
import json
import hashlib
import gc
import time
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '5'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
from scipy import sparse, linalg
from scipy.special import softmax, log_softmax, logsumexp
from scipy.optimize import minimize
import torch
from torch import nn
from torch.nn import functional as F
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TRAIN_BATCH_SIZE = 128
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True)
from tqdm.auto import tqdm
from functools import partial
SHOW_PROGRESS = True
progress = partial(tqdm, disable=not SHOW_PROGRESS, mininterval=1.0, dynamic_ncols=True)
CURRENT_TRAINING = 'Трансформер'
pipeline_progress = progress(total=8)
pipeline_started = time.perf_counter()

def finish_stage():
    pipeline_progress.update(1)

def parquet_progress(parquet, batch_size, columns=None):
    batches = parquet.iter_batches(batch_size=batch_size, columns=columns)
    with progress(total=parquet.metadata.num_rows, leave=False) as bar:
        for batch in batches:
            yield batch
            bar.update(batch.num_rows)

def row_progress(stream, total, size):
    with progress(total=total, leave=False) as bar:
        for item in stream:
            yield item
            bar.update(size(item))

def train_booster_progress(params, dataset, num_boost_round):
    with progress(total=num_boost_round, leave=False) as bar:

        def update(environment):
            bar.update(1)
        update.order = 50
        update.before_iteration = False
        return lgb.train(params, dataset, num_boost_round=num_boost_round, callbacks=[update])
DATA_DIR = Path('.').resolve()
OUTPUT_DIR = Path('.').resolve()
os.chdir(DATA_DIR)
SEED = 777
SEEDS = [SEED]
CLASSES = None
START = pd.Timestamp('2026-04-07', tz='UTC').value // 1000000


  0%|          | 0/8 [00:00<?, ?it/s]

## 2. Загрузка данных


In [ ]:
train_labels = pq.read_table('train.parquet', columns=['target_product'])[0].to_numpy()
CLASSES = np.unique(train_labels)
event_dictionary = pd.read_parquet('event_dictionary.parquet')
sample_submission = pd.read_csv('sample_submission.csv', dtype={'example_id': str})
finish_stage()


## 3. Feature engineering 1

In [ ]:
market_HOURS6 = 21600000

def market_build_market(pf, target_lookup):
    counts = {}
    buckets = 5
    for batch in parquet_progress(pf, 512, columns=['client_id', 'events']):
        lists = batch.column('events')
        flat = lists.flatten()
        e = flat.field('encoded_event').to_numpy()
        p = flat.field('encoded_product').fill_null(-1).to_numpy()
        mask = (p >= 0) & ((e == 13) | target_lookup[e])
        t = flat.field('timestamp').to_numpy()[mask] // market_HOURS6
        group = np.repeat(batch.column('client_id').to_numpy() % buckets, lists.value_lengths().to_numpy())[mask]
        packed = ((t * 5 + group) * 2 + target_lookup[e[mask]]) * 128 + p[mask]
        keys, values = np.unique(packed, return_counts=True)
        for k, v in zip(keys.tolist(), values.tolist()):
            counts[k] = counts.get(k, 0) + v
    keys = np.array(list(counts), dtype=np.int64)
    bucket_time, product = np.divmod(keys, 128)
    bucket_time, action = np.divmod(bucket_time, 2)
    time, bucket = np.divmod(bucket_time, 5)
    first = time.min()
    arr = np.zeros((time.max() - first + 1, 5, 2, 128), dtype=np.int64)
    arr[time - first, bucket, action, product] = list(counts.values())
    arr = arr.sum(axis=1, keepdims=True) - arr
    return (first, np.pad(arr.cumsum(axis=0), ((1, 0), (0, 0), (0, 0), (0, 0))))

def market_market_features(market, times, clients, classes):
    first, totals = market
    end = times // market_HOURS6 - first
    hi = end.clip(0, len(totals) - 1)
    features = {}
    for lag in (1, 4, 12, 40):
        lo = (end - lag).clip(0, len(totals) - 1)
        counts = totals[hi, clients % 5] - totals[lo, clients % 5]
        views, clicks = (counts[:, 0, classes], counts[:, 1, classes])
        for tag, a in (('view', views), ('click', clicks)):
            features[f'market_{tag}_{lag}'] = np.log((a + 0.5) / (a.sum(1, keepdims=True) + len(classes) * 0.5))
        features[f'market_ctr_{lag}'] = np.log((clicks + 0.5) / (views + 10))
    return features
features_OUT = Path('artifacts')
features_DAY = 86400000
features_PROXY_START = pd.Timestamp('2026-04-08', tz='UTC').value // 1000000

def features_grouped_features(e, p, t, offsets, classes, lookup, market=None, clients=None):
    size, width = (len(offsets) - 1, int(classes.max()) + 2)
    lengths = np.diff(offsets)
    row = np.repeat(np.arange(size), lengths)
    last = offsets[1:] - 1
    pos = np.arange(len(e))
    age = last[row] - pos
    minutes = (t[last[row]] - t) / 60000.0
    pair = row * width + np.where((p >= 0) & (p < width - 1), p, width - 1)
    shape = (size, width)
    grid = size * width
    features = {}

    def add(key, matrix):
        features[key] = np.asarray(matrix, dtype=np.float32)

    def count(mask, weights=None):
        return np.bincount(pair[mask], weights=None if weights is None else weights[mask], minlength=grid).reshape(shape)[:, classes]

    def counts(key, mask):
        x = count(mask)
        add(key, np.log1p(x))
        return x

    def nearest(key, mask):
        idx = np.full(grid, -1, dtype=np.int32)
        np.maximum.at(idx, pair[mask], pos[mask])
        idx = idx.reshape(shape)[:, classes]
        add(key + '_events', np.where(idx >= 0, np.log1p(last[:, None] - idx), 12))
        delta = np.maximum(0, t[last, None] - t[idx]) / 60000
        add(key + '_minutes', np.where(idx >= 0, np.log1p(delta), 14))
        return idx

    def broadcast(key, values):
        add(key, np.broadcast_to(values[:, None], (size, len(classes))))
    target = lookup[0][e]
    click = lookup[1][e]
    view = e == 13
    known = p >= 0
    prev = np.roll(e, 1)
    prev[offsets[:-1]] = 0
    add('product', np.broadcast_to(classes, (size, len(classes))))
    shown = counts('views', view)
    clicked = counts('target_clicks', target)
    add('personal_ctr', np.log1p(clicked) - np.log1p(shown / 4))
    for key, mask in (('view', view), ('target', target), ('other', known & ~view & ~target), ('other_click', click & ~target & known), ('any', known)):
        nearest(key, mask)
    for n in (5, 15, 40, 120, 400):
        x = counts('views_last_' + str(n), view & (age < n))
        add('view_share_' + str(n), x / np.maximum(1, x.sum(axis=1, keepdims=True)))
    for m in (5, 60, 1440, 10080):
        counts('views_minutes_' + str(m), view & (minutes <= m))
    counts('recent_target', target & (age < 150))
    counts('other_click_count', known & click & ~target)
    for code in (399, 542, 835, 666):
        counts('event_' + str(code), e == code)
    for before in (962, 234, 80, 13, 971):
        mask = view & (prev == before)
        counts('after_' + str(before), mask)
        nearest('after_' + str(before), mask)
    for n in (20, 150):
        add('decay_' + str(n), np.log1p(count(view, np.exp2(-age / n))))
    for name, mask in (('last_product', known), ('last_banner', view), ('last_target', target & known)):
        idx = np.full(size, -1)
        np.maximum.at(idx, row[mask], pos[mask])
        broadcast(name, np.where(idx >= 0, p[idx], -1))
    broadcast('history_n', np.log1p(lengths))
    broadcast('history_minutes', np.log1p((t[last] - t[offsets[:-1]]) / 60000))
    broadcast('distinct_views', (shown > 0).sum(1))
    broadcast('sum_views', np.log1p(shown.sum(1)))
    broadcast('sum_target', np.log1p(clicked.sum(1)))
    broadcast('end_hour', t[last] // 3600000 % 24)
    for j in (0, 1, 2, 4):
        broadcast('end_event_' + str(j), e[np.maximum(last - j, offsets[:-1])])
    if market is not None:
        features.update(market_market_features(market, t[last], clients, classes))
        match = np.ones(len(e), dtype=bool)
        for j in range(4):
            match &= (pos - offsets[row] >= j) & (np.roll(e, j) == e[np.maximum(last[row] - j, offsets[row])])
            if j == 2:
                continue
            last_match = np.maximum.accumulate(np.where(match, pos, -100000))
            prev_match = np.roll(last_match, 1)
            distance = pos - prev_match
            nearby = (prev_match >= offsets[row]) & (distance > 0) & (distance <= 6)
            counts(f'suffix_{j + 1}_views', view & nearby)
            counts(f'suffix_{j + 1}_other', known & ~view & nearby)
            weights = np.exp2(-age / 100)
            add(f'suffix_{j + 1}_recent', np.log1p(count(view & nearby, weights)))
        for code in (399, 542, 835, 666):
            nearest(f'detail_{code}', e == code)
        for step in (1, 2, 3, 4):
            anchor = np.roll(e, step)
            counts(f'anchor_{step}', view & (anchor == 962) & (pos - offsets[row] >= step))
        counts('view_older', view & (minutes > 1440))
        counts('view_middle', view & (minutes > 60) & (minutes <= 1440))
        counts('target_7d', target & (minutes <= 10080))
        burst = view & (prev != 13)
        counts('burst_first', burst)
        nearest('burst_first', burst)
    return (np.stack(list(features.values()), axis=-1).astype(np.float16), list(features))

def features_build(rich=False):
    directory = features_OUT / 'rich' if rich else features_OUT
    directory.mkdir(exist_ok=True, parents=True)
    labels = pq.read_table('train.parquet', columns=['target_product'])[0].to_numpy()
    classes = np.unique(labels)
    np.save(directory / 'classes.npy', classes)
    d = pd.read_parquet('event_dictionary.parquet').set_index('encoded_event')
    idx = np.arange(d.index.max() + 1)
    lookup = (d.is_target_event.reindex(idx, fill_value=False).to_numpy(), d.event_type.reindex(idx).eq('click').to_numpy())
    proxy_x, proxy_meta = ([], [])
    for name in ('train', 'test'):
        pf = pq.ParquetFile(name + '.parquet')
        market = market_build_market(pf, lookup[0]) if rich else None
        out = None
        offset = 0
        meta = []
        for b in parquet_progress(pf, 256):
            lists = b.column('events')
            flat = lists.flatten()
            cuts = lists.offsets.to_numpy()
            t = flat.field('timestamp').to_numpy()
            e = flat.field('encoded_event').to_numpy()
            p = flat.field('encoded_product').fill_null(-1).to_numpy()
            clients = b.column('client_id').to_numpy()
            x, names = features_grouped_features(e, p, t, cuts, classes, lookup, market, clients)
            if out is None:
                out = np.lib.format.open_memmap(directory / (name + '.npy'), mode='w+', dtype='float16', shape=(pf.metadata.num_rows, *x.shape[1:]))
            out[offset:offset + len(b)] = x
            meta.append(pd.DataFrame(dict(example_id=b.column('example_id').to_pylist(), client=clients, end=t[cuts[1:] - 1], length=np.diff(cuts))))
            if name == 'test':
                for i, (a, z) in enumerate(zip(cuts, cuts[1:])):
                    take = np.flatnonzero(lookup[0][e[a:z]] & (p[a:z] >= 0) & (t[a:z] >= features_PROXY_START))
                    for j in take[::-1]:
                        end = a + j - 5
                        if end - a < 20 or lookup[0][e[end:end + 5]].any():
                            continue
                        sample, _ = features_grouped_features(e[a:end], p[a:end], t[a:end], np.array([0, end - a]), classes, lookup, market, clients[i:i + 1])
                        proxy_x.append(sample[0])
                        proxy_meta.append((clients[i], t[a + j], p[a + j], end - a))
                        break
            offset += len(b)
        del out
        pd.concat(meta, ignore_index=True).to_pickle(directory / (name + '_meta.pkl'))
    np.save(directory / 'proxy.npy', np.stack(proxy_x))
    pd.DataFrame(proxy_meta, columns=['client', 'end', 'target', 'length']).to_pickle(directory / 'proxy_meta.pkl')
    (directory / 'features.txt').write_text('\n'.join(names))

def augment_examples(pf, classes, target_event):
    valid_product = np.isin(np.arange(128), classes)
    for b in parquet_progress(pf, 256):
        lists = b.column('events')
        flat = lists.flatten()
        e = flat.field('encoded_event').to_numpy()
        p = flat.field('encoded_product').fill_null(-1).to_numpy()
        t = flat.field('timestamp').to_numpy()
        offsets = lists.offsets.to_numpy()
        anchors = (e == target_event) & (p >= 0) & valid_product[p]
        chosen = []
        for i, (a, z) in enumerate(zip(offsets, offsets[1:])):
            for j in np.flatnonzero(anchors[a:z])[::-1]:
                if j >= 25 and (not (e[a + j - 5:a + j] == target_event).any()):
                    chosen.append((i, a, a + j - 5, a + j))
                    break
        if chosen:
            i, a, z, j = np.asarray(chosen).T
            mask = np.zeros(len(e) + 1, dtype=int)
            np.add.at(mask, a, 1)
            np.add.at(mask, z, -1)
            keep = mask.cumsum()[:-1].astype(bool)
            cuts = np.r_[0, (z - a).cumsum()]
            clients = b.column('client_id').to_numpy()[i]
            yield (e[keep], p[keep], t[keep], cuts, pd.DataFrame(dict(client=clients, target=p[j], end=t[z - 1], source=b.column('example_id').to_numpy(zero_copy_only=False)[i])))

def augment_main():
    directory = features_OUT / 'rich'
    classes = np.load(directory / 'classes.npy')
    d = pd.read_parquet('event_dictionary.parquet').set_index('encoded_event')
    target_event, = d.index[d.is_target_event]
    idx = np.arange(d.index.max() + 1)
    lookup = (d.is_target_event.reindex(idx, fill_value=False).to_numpy(), d.event_type.reindex(idx).eq('click').to_numpy())
    pf = pq.ParquetFile('train.parquet')
    market = market_build_market(pf, lookup[0])
    names = (directory / 'features.txt').read_text().splitlines()
    n = sum((len(b[-1]) for b in augment_examples(pf, classes, target_event)))
    data = np.lib.format.open_memmap(directory / 'aux.npy', mode='w+', dtype='float16', shape=(n, len(classes), len(names)))
    meta = []
    offset = 0
    for e, p, t, cuts, rows in row_progress(augment_examples(pf, classes, target_event), n, lambda item: len(item[-1])):
        x, cols = features_grouped_features(e, p, t, cuts, classes, lookup, market, rows.client.to_numpy())
        assert cols == names
        data[offset:offset + len(rows)] = x
        meta.append(rows)
        offset += len(rows)
    pd.concat(meta, ignore_index=True).to_pickle(directory / 'aux_meta.pkl')


## 4. Feature engineering 2


In [ ]:
choice_features_V2 = features_OUT / 'choice'
choice_features_RICH = Path(os.environ['RICH_DIR']) if os.environ.get('RICH_DIR') else features_OUT / 'rich'

def choice_features_top_actions(pf):
    freq = np.zeros(1100, dtype=np.int64)
    for b in parquet_progress(pf, 512, columns=['events']):
        flat = b[0].flatten()
        e = flat.field('encoded_event').to_numpy()
        missing = flat.field('encoded_product').is_null().to_numpy(zero_copy_only=False)
        freq += np.bincount(e[missing], minlength=len(freq))
    return np.sort(np.argsort(-freq, kind='stable')[:160])

def choice_features_extract(e, p, t, offsets, classes, actions):
    n = len(offsets) - 1
    row = np.repeat(np.arange(n), np.diff(offsets))
    pos = np.arange(len(e))
    end = offsets[1:] - 1
    age = end[row] - pos
    hours = (t[end[row]] - t) / 3600000.0
    known = p >= 0
    product = np.where(known, p, 70)
    pair = row * 71 + product
    features = {}
    context = []

    def product_count(mask, weight=None):
        return np.bincount(pair[mask], weights=None if weight is None else weight[mask], minlength=n * 71).reshape(n, 71)

    def add(name, mask, weight=None):
        values = product_count(mask, weight)[:, classes]
        features[name] = np.log1p(values)
        return values

    def event_counts(mask):
        return np.bincount((row * 1100 + e)[mask], minlength=n * 1100).reshape(n, 1100)[:, actions]
    gap = np.r_[0, np.diff(t)]
    boundary = gap > 30 * 60000
    boundary[offsets[:-1]] = True
    session = boundary.cumsum() - 1
    current = session == session[end[row]]
    session_age = session[end[row]] - session
    view = (e == 13) | (e == 399)
    click = e == 469
    for tag, mask in (('banner', e == 13), ('alternate', e == 399), ('click', click)):
        idx = np.flatnonzero(mask & known)
        if len(idx):
            _, first = np.unique(session[idx] * 71 + product[idx], return_index=True)
            idx = idx[first]
        once = np.zeros(len(e), dtype=bool)
        once[idx] = True
        distinct = add(tag + '_sessions', once)
        add(tag + '_sessions_3d', once & (hours < 72))
        recent = add(tag + '_sess_decay', once, np.exp2(-session_age / 3))
        features[tag + '_session_share'] = recent / (0.5 + recent.sum(1, keepdims=True))
        add(tag + '_this_session', mask & current)
        if tag != 'click':
            total = product_count(mask)[:, classes]
            features[tag + '_repeats'] = np.log1p(total) - np.log1p(distinct)
    for code in (13, 399):
        mask = e == code
        idx = np.flatnonzero(mask & known)
        first = np.zeros(len(e), bool)
        last = np.zeros(len(e), bool)
        if len(idx):
            ss = session[idx]
            first[idx[np.r_[True, ss[1:] != ss[:-1]]]] = True
            last[idx[np.r_[ss[1:] != ss[:-1], True]]] = True
        add(f'first_{code}', first, np.exp2(-session_age / 4))
        add(f'last_{code}', last, np.exp2(-session_age / 4))
    for window in (24, 168):
        num = product_count(click & (hours < window))[:, classes]
        denom = product_count(view & (hours < window))[:, classes]
        features[f'personal_log_ctr_{window}'] = np.log((num + 0.35) / (denom + 12))
    indices = np.flatnonzero(view & known)
    switch = np.zeros(len(e), bool)
    if len(indices):
        prev = np.r_[indices[0], indices[:-1]]
        switch[indices] = (p[indices] != p[prev]) | (session[indices] != session[prev])
        switch[indices[0]] = True
    add('distinct_run', switch, np.exp2(-age / 90))
    latest = np.full(n, -1, dtype=int)
    np.maximum.at(latest, row[indices], indices)
    follows = np.zeros(len(e), bool)
    if len(indices) > 1:
        a, b = (indices[:-1], indices[1:])
        follows[b] = (session[a] == session[b]) & (p[a] == p[latest[row[b]]])
    add('follow_last', follows, np.exp2(-age / 180))
    for step in (1, 2, 3):
        within = pos - offsets[row] >= step
        mask = (e == 13) & within & (np.roll(e, step) == 13)
        add(f'carousel_{step}', mask, np.exp2(-age / 100))
    for mask in (np.ones(len(e), bool), age < 60, current):
        context.append(np.log1p(event_counts(mask)))
    for mask in (view, view & (age < 60), click, known & ~view & ~click):
        context.append(np.log1p(product_count(mask)[:, :70]))
    for j in (0, 1, 2):
        last_code = e[np.maximum(end - j, offsets[:-1])]
        context.append(last_code[:, None] == actions[None, :])
    context.append(np.column_stack([np.log1p(np.diff(offsets)), np.log1p(session[end] - session[offsets[:-1]]), np.log1p(np.maximum(0, gap[end]) / 1000), np.log1p((t[end] - t[offsets[:-1]]) / 60000)]))
    return (np.stack(list(features.values()), axis=-1).astype(np.float16), np.hstack(context).astype(np.float16), list(features))

def choice_features_batches(pf):
    for b in parquet_progress(pf, 256):
        lists = b.column('events')
        flat = lists.flatten()
        yield (flat.field('encoded_event').to_numpy(), flat.field('encoded_product').fill_null(-1).to_numpy(), flat.field('timestamp').to_numpy(), lists.offsets.to_numpy(), b.column('client_id').to_numpy())

def choice_features_build():
    choice_features_V2.mkdir(exist_ok=True, parents=True)
    classes = np.load(choice_features_RICH / 'classes.npy')
    pf = pq.ParquetFile('train.parquet')
    actions = choice_features_top_actions(pf)
    proxy_x, proxy_c, proxy_ids = ([], [], [])
    for name in ('train', 'test', 'aux'):
        if name == 'aux':
            stream = ((e, p, t, o, m.client.to_numpy()) for e, p, t, o, m in augment_examples(pf, classes, 469))
        else:
            stream = choice_features_batches(pq.ParquetFile(name + '.parquet'))
        meta = pd.read_pickle(choice_features_RICH / (name + '_meta.pkl'))
        i = 0
        exp_i = 0
        exp_meta = []
        rng = np.random.default_rng(461)
        for e, p, t, o, clients in stream:
            x, c, names = choice_features_extract(e, p, t, o, classes, actions)
            if i == 0:
                prod = np.lib.format.open_memmap(choice_features_V2 / (name + '.npy'), 'w+', dtype='float16', shape=(len(meta), *x.shape[1:]))
                ctx = np.lib.format.open_memmap(choice_features_V2 / (name + '_context.npy'), 'w+', dtype='float16', shape=(len(meta), c.shape[1]))
                if name == 'train':
                    exposure = np.lib.format.open_memmap(choice_features_V2 / 'exposure_context.npy', 'w+', dtype='uint8', shape=(len(meta), c.shape[1]))
            assert np.array_equal(clients, meta.client.to_numpy()[i:i + len(c)])
            prod[i:i + len(c)] = x
            ctx[i:i + len(c)] = c
            if name == 'train':
                chosen = []
                for k, (a, z) in enumerate(zip(o, o[1:])):
                    valid = np.flatnonzero(np.isin(e[a:z], [13, 399]) & np.isin(p[a:z], classes))
                    valid = valid[valid >= 24]
                    if len(valid):
                        j = a + rng.choice(valid[-24:])
                        chosen.append((k, a, j - 4, j))
                if chosen:
                    k, a, z, j = np.asarray(chosen).T
                    marks = np.zeros(len(e) + 1, dtype=int)
                    np.add.at(marks, a, 1)
                    np.add.at(marks, z, -1)
                    keep = marks.cumsum()[:-1].astype(bool)
                    _, cx, _ = choice_features_extract(e[keep], p[keep], t[keep], np.r_[0, (z - a).cumsum()], classes, actions)
                    exposure[exp_i:exp_i + len(cx)] = np.clip(np.rint(cx * 16), 0, 255).astype(np.uint8)
                    exp_i += len(cx)
                    exp_meta.append(pd.DataFrame(dict(client=clients[k], target=p[j], end=t[z - 1])))
            if name == 'test':
                for client, a, z in zip(clients, o, o[1:]):
                    clicks = np.flatnonzero((e[a:z] == 469) & (p[a:z] >= 0) & (t[a:z] >= features_PROXY_START))
                    for j in clicks[::-1]:
                        end = a + j - 5
                        if end - a >= 20 and (not (e[end:end + 5] == 469).any()):
                            px, pc, _ = choice_features_extract(e[a:end], p[a:end], t[a:end], np.array([0, end - a]), classes, actions)
                            proxy_x.append(px[0])
                            proxy_c.append(pc[0])
                            proxy_ids.append(client)
                            break
            i += len(c)
        assert i == len(meta)
        del prod, ctx
        if name == 'train':
            del exposure
            pd.concat(exp_meta, ignore_index=True).to_pickle(choice_features_V2 / 'exposure_meta.pkl')
    np.save(choice_features_V2 / 'proxy.npy', np.stack(proxy_x))
    np.save(choice_features_V2 / 'proxy_context.npy', np.stack(proxy_c))
    assert np.array_equal(proxy_ids, pd.read_pickle(choice_features_RICH / 'proxy_meta.pkl').client)
    (choice_features_V2 / 'names.txt').write_text('\n'.join(names))

def routes_extract(e, p, t, o, classes):
    n = len(o) - 1
    row = np.repeat(np.arange(n), np.diff(o))
    pos = np.arange(len(e))
    last = o[1:] - 1
    start = o[row]
    end = last[row]
    age = end - pos
    dt = np.maximum(0, np.r_[0, np.diff(t)])
    dt[o[:-1]] = 0
    known = p >= 0
    main = e == 13
    ad = (main | (e == 399)) & known
    prev = np.maximum.accumulate(np.where(~main, pos, -1))
    depth = pos - np.maximum(prev, start - 1)
    anchor = np.where(prev >= start, e[prev], 0)
    rules = []
    names = []

    def rule(name, code):
        names.append(name)
        rules.append(code)
    for tag, mask in [('main', main), ('alt', e == 399), ('clicked', e == 469), ('nonad', ~ad & (e != 469)), ('home', main & (anchor == 962)), ('screen234', main & (anchor == 234)), ('screen80', main & (anchor == 80)), ('first', main & (depth == 1)), ('second', main & (depth == 2)), ('later', main & (depth >= 3)), ('homefirst', main & (depth == 1) & (anchor == 962)), ('homelater', main & (depth >= 2) & (anchor == 962))]:
        mask = mask & known
        ix = np.full(n, -1, dtype=int)
        np.maximum.at(ix, row[mask], pos[mask])
        first = np.where(ix >= 0, p[ix], -1)
        rule(tag + '_last', first)
        mask2 = mask & (p != first[row])
        ix.fill(-1)
        np.maximum.at(ix, row[mask2], pos[mask2])
        rule(tag + '_previous_distinct', np.where(ix >= 0, p[ix], -1))
        w = np.exp2(-age[mask] / 90)
        cnt = np.bincount(row[mask] * 71 + p[mask], weights=w, minlength=n * 71).reshape(n, 71)
        top = cnt.argmax(1)
        rule(tag + '_usual', np.where(cnt.max(1) > 0, top, -1))
    length = np.zeros(len(e), int)
    length_prod = length.copy()
    m = np.ones(len(e), bool)
    mp = m.copy()
    for k in range(7):
        a = np.maximum(start, pos - k)
        b = np.maximum(start, end - k)
        m &= (pos >= start + k) & (e[a] == e[b])
        mp &= m & (p[a] == p[b])
        length += m
        length_prod += mp
    last_ad = np.maximum.accumulate(np.where(ad, pos, -1))
    next_ad = np.minimum.accumulate(np.where(ad, pos, len(e))[::-1])[::-1]
    eligible = age >= 5
    future = np.minimum(end, pos + 5)
    nearest = last_ad[future]
    first = next_ad[np.minimum(end, pos + 1)]
    after = next_ad[future]
    look = []
    for tag, ix, valid in [('by_fifth', nearest, nearest >= np.maximum(start, pos - 1)), ('first_next', first, first <= future), ('fifth_or_next', after, after <= np.minimum(end, pos + 9))]:
        good = eligible & valid & (ix >= 0) & (ix < len(e))
        for mode, matched in [('event', length), ('both', length_prod)]:
            use = good & (matched >= 1)
            w = (np.exp2(matched[use] * 1.5) - 1) * np.exp2(-age[use] / 300)
            rapid = (dt[use] < 1600) == (dt[end[use]] < 1600)
            w *= np.where(rapid, 1.0, 0.45)
            counts = np.bincount(row[use] * 71 + p[ix[use]], weights=w, minlength=n * 71).reshape(n, 71)
            selected = counts[:, classes]
            look.append(selected / (2 + counts.sum(1, keepdims=True)))
            rule(tag + '_' + mode + '_winner', np.where(counts.max(1) > 0, counts.argmax(1), -1))
    clock = [np.log1p(dt[last - k] / 1000) for k in range(6)]
    clock.extend([np.log1p((t[last] - t[last - k]) / 1000) for k in (5, 15)])
    clock.extend([e[last - k] for k in (3, 5, 6, 7)])
    clock.extend([(t[last] // 86400000 + 3) % 7, t[last] // 60000 % 60])
    return (np.column_stack(rules).astype(np.int8), np.stack(look, axis=-1).astype(np.float16), np.column_stack(clock).astype(np.float16), names)

def routes_build():
    classes = np.load(choice_features_RICH / 'classes.npy')
    proxy = [[], [], []]
    for name in ('train', 'test', 'aux'):
        pf = pq.ParquetFile(('train' if name == 'aux' else name) + '.parquet')
        if name == 'aux':
            stream = ((e, p, t, o, m.client.to_numpy()) for e, p, t, o, m in augment_examples(pf, classes, 469))
        else:
            stream = choice_features_batches(pf)
        meta = pd.read_pickle(choice_features_RICH / (name + '_meta.pkl'))
        offset = 0
        for e, p, t, o, clients in stream:
            *items, names = routes_extract(e, p, t, o, classes)
            if offset == 0:
                arrays = [np.lib.format.open_memmap(choice_features_V2 / f'{name}_route{k}.npy', 'w+', dtype=x.dtype, shape=(len(meta), *x.shape[1:])) for k, x in enumerate(items)]
            for dest, x in zip(arrays, items):
                dest[offset:offset + len(clients)] = x
            assert np.array_equal(clients, meta.client.to_numpy()[offset:offset + len(clients)])
            offset += len(clients)
            if name == 'test':
                for a, z in zip(o, o[1:]):
                    for j in np.flatnonzero((e[a:z] == 469) & (p[a:z] >= 0) & (t[a:z] >= features_PROXY_START))[::-1]:
                        if j >= 25 and (not (e[a + j - 5:a + j] == 469).any()):
                            cut = a + j - 5
                            for dst, x in zip(proxy, routes_extract(e[a:cut], p[a:cut], t[a:cut], np.array([0, cut - a]), classes)):
                                dst.append(x[0])
                            break
        assert offset == len(meta)
        del arrays
    for k, data in enumerate(proxy):
        np.save(choice_features_V2 / f'proxy_route{k}.npy', np.stack(data))
    (choice_features_V2 / 'route_names.txt').write_text('\n'.join(names))

def observed_clicks_stream():
    for b in parquet_progress(pq.ParquetFile('test.parquet'), 32):
        lists = b.column('events')
        flat = lists.flatten()
        e = flat.field('encoded_event').to_numpy()
        p = flat.field('encoded_product').fill_null(-1).to_numpy()
        t = flat.field('timestamp').to_numpy()
        o = lists.offsets.to_numpy()
        clients = b.column('client_id').to_numpy()
        ids = b.column('example_id').to_numpy(zero_copy_only=False)
        pieces, rows = ([], [])
        for ix, (a, z) in enumerate(zip(o, o[1:])):
            for j in np.flatnonzero((e[a:z] == 469) & (p[a:z] >= 0)):
                if j < 25 or (e[a + j - 5:a + j] == 469).any():
                    continue
                end = a + j - 5
                assert end > a and end + 5 == a + j
                assert t[end - 1] <= t[a + j] <= t[z - 1]
                pieces.append(np.arange(a, end))
                rows.append((clients[ix], p[a + j], t[end - 1], t[a + j], ids[ix], end - a, int(j)))
        if pieces:
            take = np.concatenate(pieces)
            cuts = np.r_[0, np.cumsum([len(x) for x in pieces])]
            meta = pd.DataFrame(rows, columns=['client', 'target', 'end', 'click_time', 'source', 'length', 'click_position'])
            yield (e[take], p[take], t[take], cuts, meta)

def observed_clicks_build():
    classes = np.load(choice_features_RICH / 'classes.npy')
    n = sum((len(m) for e, p, t, o, m in observed_clicks_stream()))
    d = pd.read_parquet('event_dictionary.parquet').set_index('encoded_event')
    idx = np.arange(d.index.max() + 1)
    lookup = (d.is_target_event.reindex(idx, fill_value=False).to_numpy(), d.event_type.reindex(idx).eq('click').to_numpy())
    market = market_build_market(pq.ParquetFile('test.parquet'), lookup[0])
    actions = choice_features_top_actions(pq.ParquetFile('train.parquet'))
    paths = [choice_features_RICH / 'observed.npy', choice_features_V2 / 'observed.npy', choice_features_V2 / 'observed_context.npy', *[choice_features_V2 / f'observed_route{k}.npy' for k in range(3)]]
    arrays = None
    metadata = []
    start = 0
    for e, p, t, o, m in row_progress(observed_clicks_stream(), n, lambda item: len(item[-1])):
        rich, names = features_grouped_features(e, p, t, o, classes, lookup, market, m.client.to_numpy())
        extra, context, _ = choice_features_extract(e, p, t, o, classes, actions)
        values = [rich, extra, context, *routes_extract(e, p, t, o, classes)[:3]]
        if arrays is None:
            assert names == (choice_features_RICH / 'features.txt').read_text().splitlines()
            arrays = [np.lib.format.open_memmap(path, 'w+', dtype=x.dtype, shape=(n, *x.shape[1:])) for path, x in zip(paths, values)]
        for dst, x in zip(arrays, values):
            dst[start:start + len(m)] = x
        metadata.append(m)
        start += len(m)
    assert start == n
    del arrays
    meta = pd.concat(metadata, ignore_index=True)
    meta['allowed_class'] = np.isin(meta.target, classes)
    meta['client_fold'] = meta.client % 5
    assert not meta.duplicated(['client', 'click_position']).any()
    assert (meta.click_time >= meta.end).all()
    meta.to_pickle(choice_features_RICH / 'observed_meta.pkl')
extra_obs_MULT = None

def regime_mult(end):
    return 1.0
features_OUT = Path('sub24_private_stable_seed777_artifacts')
choice_features_V2 = features_OUT / 'choice'
choice_features_RICH = features_OUT / 'rich'
BASE_RICH = choice_features_RICH
CAMP_RICH = features_OUT / 'rich_camp6'
for directory in (features_OUT, choice_features_V2, BASE_RICH, CAMP_RICH):
    directory.mkdir(parents=True, exist_ok=True)


## 5. Feature engineering 2


In [ ]:
campaign_activity_v8_HOUR = 3600000

def campaign_activity_v8_build():
    campaign_activity_v8_ROOT.mkdir(exist_ok=True)
    train_clients = pq.read_table('train.parquet', columns=['client_id'])[0].to_numpy()
    test_clients = pq.read_table('test.parquet', columns=['client_id'])[0].to_numpy()
    overlap = set(np.intersect1d(train_clients, test_clients).tolist())
    seen = set()
    pieces = []
    for source in ['train', 'test']:
        for batch in parquet_progress(pq.ParquetFile(source + '.parquet'), 256, columns=['client_id', 'events']):
            lists = batch.column('events')
            flat = lists.flatten()
            o = lists.offsets.to_numpy()
            e = flat.field('encoded_event').to_numpy()
            p = flat.field('encoded_product').fill_null(-1).to_numpy()
            t = flat.field('timestamp').to_numpy()
            clients = np.repeat(batch.column('client_id').to_numpy(), np.diff(o))
            mask = (p >= 0) & np.isin(e, [13, 399, 469])
            ix = np.flatnonzero(mask)
            duplicate = []
            for j in ix[np.isin(clients[ix], list(overlap))]:
                key = (int(clients[j]), int(t[j]), int(e[j]), int(p[j]))
                if source == 'test' and key in seen:
                    duplicate.append(j)
                if source == 'train':
                    seen.add(key)
            mask[duplicate] = False
            ix = np.flatnonzero(mask)
            typ = np.searchsorted(np.array([13, 399, 469]), e[ix])
            packed = ((t[ix] // campaign_activity_v8_HOUR * 5 + clients[ix] % 5) * 3 + typ) * 128 + p[ix]
            keys, counts = np.unique(packed, return_counts=True)
            pieces.append((keys, counts))
    keys, counts = map(np.concatenate, zip(*pieces))
    unique, inv = np.unique(keys, return_inverse=True)
    counts = np.bincount(inv, weights=counts)
    z, prod = np.divmod(unique, 128)
    z, typ = np.divmod(z, 3)
    hour, group = np.divmod(z, 5)
    first = int(hour.min())
    arr = np.zeros((int(hour.max() - first + 1), 5, 3, 128), np.float64)
    arr[hour - first, group, typ, prod] = counts
    np.savez(campaign_activity_v8_ROOT / 'counts.npz', first=first, counts=arr)
campaign_activity_v8_ROOT = features_OUT / 'campaign_counts'

@lru_cache(maxsize=1)
def campaign_cumulative():
    saved = np.load(campaign_activity_v8_ROOT / 'counts.npz')
    counts = saved['counts']
    counts = counts.sum(1, keepdims=True) - counts
    return (int(saved['first']), np.pad(counts.cumsum(0), ((1, 0), (0, 0), (0, 0), (0, 0))))

@lru_cache(maxsize=8)
def campaign_features(name):
    meta = pd.read_pickle(BASE_RICH / (name + '_meta.pkl'))
    first, totals = campaign_cumulative()
    hour = (meta.end.to_numpy() // 3600000 - first).astype(int)
    group = meta.client.to_numpy() % 5
    blocks = []
    for width in (6, 24, 96):
        lo = np.clip(hour - width, 0, len(totals) - 1)
        hi = np.clip(hour + width + 1, 0, len(totals) - 1)
        counts = (totals[hi, group] - totals[lo, group])[:, :, CLASSES]
        shares = np.log((counts + 1) / (counts.sum(2, keepdims=True) + len(CLASSES)))
        mix = np.log((counts[:, 0] + 1) / (counts[:, 1] + 1))
        blocks.append(np.stack([shares[:, 0], shares[:, 1], mix], axis=-1))
    return np.concatenate(blocks, axis=2).astype(np.float16)
CAMPAIGN_NAMES = [f'campaign_{h}h_{kind}' for h in (6, 24, 96) for kind in ('view13_share', 'view399_share', 'view_mix')]

def add_campaigns_to_rich():
    names = (BASE_RICH / 'features.txt').read_text().splitlines()
    (CAMP_RICH / 'features.txt').write_text('\n'.join(names + CAMPAIGN_NAMES))
    for name in ('train', 'test', 'aux', 'observed'):
        original = np.load(BASE_RICH / f'{name}.npy', mmap_mode='r')
        extra = campaign_features(name)
        result = np.lib.format.open_memmap(CAMP_RICH / f'{name}.npy', 'w+', dtype=original.dtype, shape=original.shape[:2] + (original.shape[2] + 9,))
        for i in progress(range(0, len(original), 5000), leave=False):
            result[i:i + 5000] = np.concatenate([original[i:i + 5000], extra[i:i + 5000]], axis=-1)
        result.flush()
        del result
        pd.read_pickle(BASE_RICH / f'{name}_meta.pkl').to_pickle(CAMP_RICH / f'{name}_meta.pkl')
    np.save(CAMP_RICH / 'classes.npy', CLASSES)


## 6. Последовательности для трансформера


In [ ]:
seqx_data_DROP = ['product', 'last_product', 'last_banner', 'last_target', 'end_event_0', 'end_event_1', 'end_event_2', 'end_event_4', 'end_hour']

def seqx_data_flat(name):
    tab = pq.read_table(f'{name}.parquet', columns=['example_id', 'events'])
    ev = tab.column('events').combine_chunks()
    f = ev.flatten()
    return dict(e=f.field('encoded_event').to_numpy(), p=f.field('encoded_product').fill_null(-1).to_numpy(), t=f.field('timestamp').to_numpy(), off=ev.offsets.to_numpy(), index={x: i for i, x in enumerate(tab.column('example_id').to_pylist())})

def seqx_data_tails(src, rows, z, L):
    n = len(rows)
    out = dict(ev=np.zeros((n, L), np.int16), pr=np.zeros((n, L), np.int8), dcut=np.zeros((n, L), np.float16), dprv=np.zeros((n, L), np.float16), hr=np.zeros((n, L), np.int8))
    for i, (r, zz) in progress(enumerate(zip(rows, z)), total=len(rows), leave=False):
        a = src['off'][r]
        lo = a + max(0, zz - L)
        hi = a + zz
        m = hi - lo
        e, p, t = (src['e'][lo:hi], src['p'][lo:hi], src['t'][lo:hi])
        out['ev'][i, L - m:] = e + 1
        out['pr'][i, L - m:] = np.where(p >= 0, p + 2, 1)
        out['dcut'][i, L - m:] = np.log1p((t[-1] - t) / 60000.0)
        prev = np.r_[t[0], t[:-1]] if lo == a else src['t'][lo - 1:hi - 1]
        out['dprv'][i, L - m:] = np.log1p(np.maximum(t - prev, 0) / 1000.0)
        out['hr'][i, L - m:] = (t // 3600000 + 3) % 24
    return out

def seqx_data_locate_aux(src, rows, end, target):
    z = np.full(len(rows), -1)
    for i, (r, en, tg) in progress(enumerate(zip(rows, end, target)), total=len(rows), leave=False):
        a, b = (src['off'][r], src['off'][r + 1])
        t = src['t'][a:b]
        for q in np.flatnonzero(t == en)[::-1]:
            if q + 6 < b - a and src['e'][a + q + 6] == 469 and (src['p'][a + q + 6] == tg):
                z[i] = q + 1
                break
    return z

def seqx_data_cand_features(data, n, names, keep, classes, mu=None, sd=None):
    lp, lb, lt = (names.index(c) for c in ('last_product', 'last_banner', 'last_target'))
    if mu is None:
        s = s2 = 0.0
        c = 0
        for i in progress(range(0, n, 8192), leave=False):
            x = data.rich[i:i + 8192][:, :, keep].astype(np.float64).reshape(-1, len(keep))
            s, s2, c = (s + x.sum(0), s2 + np.square(x).sum(0), c + len(x))
        mu = s / c
        sd = np.sqrt(np.maximum(s2 / c - mu ** 2, 1e-06))
    out = np.empty((n, 32, len(keep) + 3), np.float16)
    for i in progress(range(0, n, 8192), leave=False):
        r = data.rich[i:i + 8192].astype(np.float32)
        x = ((r[:, :, keep] - mu) / sd).clip(-6, 6)
        ind = np.stack([r[:, :, j] == classes[None, :] for j in (lp, lb, lt)], -1).astype(np.float32)
        out[i:i + 8192] = np.concatenate([x, ind], -1)
    return (out, mu, sd)
seqx_train_D = features_OUT / 'sequences'
seqx_train_D.mkdir(exist_ok=True)
seqx_train_DEV = DEVICE

def prepare_transformer_data():
    source = {'train': seqx_data_flat('train'), 'test': seqx_data_flat('test')}
    names = (CAMP_RICH / 'features.txt').read_text().splitlines()
    keep = [i for i, name in enumerate(names) if name not in seqx_data_DROP]
    mean = std = None
    for name in ('train', 'test', 'aux', 'observed'):
        data = choice_Data(name)
        meta = data.meta
        sequence = None
        found = np.ones(len(meta), bool)
        for source_name, raw in source.items():
            if name in ('train', 'test'):
                if source_name != name:
                    continue
                selected = np.arange(len(meta))
                rows = np.array([raw['index'][v] for v in meta.example_id])
                lengths = np.diff(raw['off'])[rows]
            else:
                selected = np.flatnonzero(meta.source.isin(list(raw['index'])).to_numpy())
                if not len(selected):
                    continue
                rows = np.array([raw['index'][v] for v in meta.source.iloc[selected]])
                lengths = seqx_data_locate_aux(raw, rows, meta.end.to_numpy()[selected], meta.target.to_numpy()[selected]) if name == 'aux' else meta.length.to_numpy()[selected]
            found[selected[lengths < 1]] = False
            tail = seqx_data_tails(raw, rows, np.maximum(lengths, 1), 256)
            if sequence is None:
                sequence = {k: np.zeros((len(meta),) + v.shape[1:], v.dtype) for k, v in tail.items()}
            for key, values in tail.items():
                sequence[key][selected] = values
        y = np.full(len(meta), -1)
        if name != 'test':
            allowed = np.isin(data.y, CLASSES)
            y[allowed] = np.searchsorted(CLASSES, data.y[allowed])
        y[~found] = -1
        np.savez(seqx_train_D / f'{name}_seq.npz', **sequence, y=y, end=meta.end.to_numpy(), client=meta.client.to_numpy(), found=found)
        candidates, mean, std = seqx_data_cand_features(data, len(meta), names, keep, CLASSES, mean, std)
        np.save(seqx_train_D / f'{name}_cand.npy', candidates)


## 7. Построение признаков из исходных файлов


In [ ]:
choice_features_RICH = BASE_RICH
features_build(rich=True)
augment_main()
choice_features_build()
routes_build()
observed_clicks_build()
campaign_activity_v8_build()
add_campaigns_to_rich()
gc.collect()
finish_stage()


  0%|          | 0/100000 [00:00<?, ?it/s]

  0%|          | 0/100000 [00:00<?, ?it/s]

  0%|          | 0/12359 [00:00<?, ?it/s]

  0%|          | 0/12359 [00:00<?, ?it/s]

  0%|          | 0/100000 [00:00<?, ?it/s]

  0%|          | 0/100000 [00:00<?, ?it/s]

... (16 outputs left out)

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

## 8. Обучение LightGBM


In [ ]:
model_SEED = SEED
model_CONFIGS = {'compact': (24, 140, 100, 35), 'deep': (48, 240, 100, 21), 'augmented': (48, 240, 100, 21), 'long': (63, 540, 120, 21)}
model_BLOCK = 1024
model_TUNE = {}

def model_zero_columns(x):
    idx = [i for i in model_TUNE.get('zero', []) if i < x.shape[-1]]
    if idx:
        x = np.array(x, copy=True)
        x[..., idx] = 0
    return x

def model_cutoff(date):
    return pd.Timestamp(date, tz='UTC').value // 1000000

def model_sample_training(data, rows, y, classes, seed):
    k = model_TUNE.get('k', 9)
    rng = np.random.default_rng(seed)
    sampled = rng.random((len(rows), len(classes)), dtype=np.float32)
    gold = np.searchsorted(classes, y[rows])
    sampled[np.arange(len(rows)), gold] = -1
    cols = np.sort(np.argpartition(sampled, k - 1, axis=1)[:, :k], axis=1)
    labels = (cols == gold[:, None]).ravel()
    matrix = model_zero_columns(data[rows[:, None], cols].reshape(len(labels), -1))
    return (matrix, labels)

def model_fit(data, rows, y, meta, classes, names, variant, directory=None):
    leaves, rounds, min_leaf, half_life = model_TUNE.get('configs', {}).get(variant, model_CONFIGS[variant])
    matrix, labels = model_sample_training(data, rows, y, classes, model_SEED)
    end = meta.end.to_numpy()
    ref = end[rows].max()
    recency = np.exp2(-((ref - end[rows]) / features_DAY) / half_life) * regime_mult(end[rows])
    if variant in {'augmented', 'long'}:
        aux, am = model_load('aux', directory)
        ids = np.flatnonzero(am.client.isin(meta.client.iloc[rows]))
        extra, extra_labels = model_sample_training(aux, ids, am.target.to_numpy(), classes, model_SEED + 1)
        matrix = np.concatenate([matrix, extra])
        labels = np.r_[labels, extra_labels]
        recency = np.r_[recency, model_TUNE.get('aux_mult', 0.55) * np.exp2(-(ref - am.end.to_numpy()[ids]) / features_DAY / half_life) * regime_mult(am.end.to_numpy()[ids])]
        del extra
    cat = [n for n in names if n == 'product' or n.startswith(('last_', 'end_event_'))]
    train = lgb.Dataset(matrix, label=labels, weight=np.repeat(recency, model_TUNE.get('k', 9)), feature_name=names, categorical_feature=cat, free_raw_data=True)
    params = dict(objective='binary', verbosity=-1, num_leaves=leaves, min_data_in_leaf=min_leaf, learning_rate=0.075, lambda_l2=8, max_bin=87, feature_fraction=0.9, deterministic=True, force_col_wise=True, num_threads=5, seed=model_SEED, feature_fraction_seed=model_SEED)
    params.update(model_TUNE.get('params', {}))
    return train_booster_progress(params, train, num_boost_round=rounds)

def model_predict(model, data, rows):
    scores = []
    for i in progress(range(0, len(rows), model_BLOCK), leave=False):
        part = model_zero_columns(data[rows[i:i + model_BLOCK]])
        pred = model.predict(part.reshape(-1, part.shape[-1]), raw_score=True, num_threads=5)
        scores.append(pred.reshape(part.shape[:2]))
    return np.vstack(scores).astype(np.float32)

def model_load(name, directory=None):
    root = directory or features_OUT
    return (np.load(root / (name + '.npy'), mmap_mode='r'), pd.read_pickle(root / (name + '_meta.pkl')))
choice_K = 12
choice_GROUPS = 4
choice_TUNE = {}
choice_FOLDS = {'march': ('2026-03-25', '2026-03-25', '2026-04-01'), 'april': ('2026-04-01', '2026-04-01', '2026-04-07'), 'late': ('2026-04-03', '2026-04-04', '2026-04-07'), 'early': ('2026-03-18', '2026-03-18', '2026-03-25'), 'w1': ('2026-03-16', '2026-03-16', '2026-03-23'), 'w2': ('2026-03-23', '2026-03-23', '2026-03-30'), 'w3': ('2026-03-30', '2026-03-30', '2026-04-16')}

def choice_array(root, name):
    return np.load(root / (name + '.npy'), mmap_mode='r')

class choice_Data:

    def __init__(self, name):
        self.name = name
        self.ctx = choice_array(choice_features_V2, name + '_context')
        self.meta = pd.read_pickle((choice_features_V2 if name == 'exposure' else choice_features_RICH) / (name + '_meta.pkl'))
        if name != 'exposure':
            self.rich = choice_array(choice_features_RICH, name)
            self.extra = choice_array(choice_features_V2, name)
        if name == 'train':
            self.y = pq.read_table('train.parquet', columns=['target_product'])[0].to_numpy()
        elif name != 'test':
            self.y = self.meta.target.to_numpy()

def choice_context_matrix(dataset, rows):
    raw = dataset.ctx[rows].astype(np.float32)
    if dataset.name == 'exposure':
        raw /= 16
    x = sparse.csr_matrix(raw)
    return sparse.hstack([x, np.ones((len(rows), 1), np.float32)], format='csr')

def choice_ridge_models(sources, classes, ref, novel_col, half_life=28):
    d = sources[0][0].ctx.shape[1] + 1
    click = sources[0][0].name != 'exposure'
    width = len(classes) * (2 if click else 1)
    cov = np.zeros((choice_GROUPS, d, d), dtype=np.float64)
    cross = np.zeros((choice_GROUPS, d, width), dtype=np.float64)
    for dataset, rows, mult in sources:
        groups = dataset.meta.client.to_numpy() % choice_GROUPS
        time = dataset.meta.end.to_numpy()
        for g in progress(range(choice_GROUPS), leave=False):
            group_rows = rows[groups[rows] == g]
            for i in range(0, len(group_rows), 2048):
                sel = group_rows[i:i + 2048]
                x = choice_context_matrix(dataset, sel)
                gold = np.searchsorted(classes, dataset.y[sel])
                y = np.zeros((len(sel), width), dtype=np.float32)
                y[np.arange(len(sel)), gold] = 1
                if click:
                    y[np.arange(len(sel)), gold + len(classes)] = dataset.rich[sel, gold, novel_col] >= 12
                weight = (np.exp2(-(ref - time[sel]) / features_DAY / half_life) * mult * (regime_mult(time[sel]) if click else 1.0)).astype(np.float32)
                if hasattr(dataset, 'sample_weight'):
                    weight *= dataset.sample_weight[sel]
                xw = x.multiply(weight[:, None]).tocsr()
                cov[g] += (x.T @ xw).toarray()
                cross[g] += xw.T @ y
    all_cov, all_cross = (cov.sum(0), cross.sum(0))
    models = []
    for g in progress(range(choice_GROUPS + 1), leave=False):
        matrix = all_cov.copy()
        rhs = all_cross.copy()
        if g < choice_GROUPS:
            matrix -= cov[g]
            rhs -= cross[g]
        matrix.flat[::d + 1] += 50.0
        models.append(linalg.solve(matrix, rhs, assume_a='pos').astype(np.float32))
    return np.stack(models)

def choice_context_scores(dataset, rows, weights, grouped):
    clicks, displays = weights
    group = dataset.meta.client.to_numpy()[rows] % choice_GROUPS if grouped else np.full(len(rows), choice_GROUPS)
    result = np.empty((len(rows), displays.shape[-1], 5), dtype=np.float32)
    for g in np.unique(group):
        ix = np.flatnonzero(group == g)
        for j in range(0, len(ix), 1024):
            indices = ix[j:j + 1024]
            x = choice_context_matrix(dataset, rows[indices])
            pos, cold = np.split(x @ (clicks[g] if clicks.ndim == 3 else clicks), 2, axis=1)
            shown = x @ (displays[g] if displays.ndim == 3 else displays)
            lift = np.log((np.maximum(0.002, pos) + 0.01) / (np.maximum(0.002, shown) + 0.02))
            result[indices] = np.stack([pos, cold, shown, lift, pos - pos.max(1, keepdims=True)], axis=-1)
    return result

def choice_batch_matrix(dataset, rows, cols, ctx):
    return np.concatenate([dataset.rich[rows[:, None], cols], dataset.extra[rows[:, None], cols], ctx[np.arange(len(rows))[:, None], cols]], axis=-1).astype(np.float32).reshape(len(rows) * cols.shape[1], -1)

def choice_train_choice(train, aux, exposure, rows, classes, names, half_life=28, additional_sources=None):
    half_life = choice_TUNE.get('half_life', half_life)
    clients = train.meta.client.iloc[rows]
    ar = np.flatnonzero(aux.meta.client.isin(clients))
    er = np.flatnonzero(exposure.meta.client.isin(clients))
    ref = train.meta.end.iloc[rows].max()
    sources = [(train, rows, 1.0), (aux, ar, choice_TUNE.get('aux_mult', 0.55))]
    sources += list(additional_sources or [])
    weights = (choice_ridge_models(sources, classes, ref, names.index('any_events'), half_life), choice_ridge_models([(exposure, er, 1.0)], classes, ref, None, half_life))
    n = sum((len(indices) for _, indices, _ in sources))
    mat = np.empty((n * choice_K, len(names)), dtype=np.float16)
    target = np.empty(n, np.int32)
    importance = np.empty(n, np.float32)
    next_row = 0
    for dataset, sel, mult in sources:
        rng = np.random.default_rng(model_SEED + {'train': 0, 'aux': 1}.get(dataset.name, 2))
        for i in progress(range(0, len(sel), 768), leave=False):
            ix = sel[i:i + 768]
            gold = np.searchsorted(classes, dataset.y[ix])
            u = rng.random((len(ix), len(classes)), dtype=np.float32)
            u[np.arange(len(ix)), gold] = -1
            cols = np.sort(np.argpartition(u, choice_K - 1, axis=1)[:, :choice_K], axis=1)
            ctx = choice_context_scores(dataset, ix, weights, True)
            end = next_row + len(ix)
            mat[next_row * choice_K:end * choice_K] = choice_batch_matrix(dataset, ix, cols, ctx)
            target[next_row:end] = (cols == gold[:, None]).argmax(1)
            importance[next_row:end] = mult * np.exp2(-(ref - dataset.meta.end.to_numpy()[ix]) / features_DAY / half_life) * regime_mult(dataset.meta.end.to_numpy()[ix])
            if hasattr(dataset, 'sample_weight'):
                importance[next_row:end] *= dataset.sample_weight[ix]
            next_row = end
    gold_rows = np.arange(n)
    binary = np.zeros((n, choice_K), dtype=np.uint8)
    binary[gold_rows, target] = 1

    def loss(raw, data):
        pred = softmax(raw.reshape(-1, choice_K), axis=1)
        hess = np.maximum(1e-05, 1.5 * pred * (1 - pred)) * importance[:, None]
        pred[gold_rows, target] -= 1
        return ((pred * importance[:, None]).ravel(), hess.ravel())
    cat = [s for s in names if s == 'product' or (s.startswith(('last_', 'end_event_')) and s in ('last_banner', 'last_product', 'last_target', 'end_event_0', 'end_event_1', 'end_event_2', 'end_event_4'))]
    params = dict(objective=loss, learning_rate=0.075, num_leaves=55, min_data_in_leaf=110, lambda_l2=6.0, max_bin=87, feature_fraction=0.9, num_threads=5, deterministic=True, force_col_wise=True, seed=model_SEED, verbosity=-1)
    params.update(choice_TUNE.get('params', {}))
    ds = lgb.Dataset(mat, label=binary.ravel(), feature_name=names, categorical_feature=cat, params={**params, 'objective': 'none'})
    ds.construct()
    del mat
    model = train_booster_progress(params, ds, num_boost_round=choice_TUNE.get('rounds', 460))
    return (model, weights)

def choice_score(model, weights, data, rows, k):
    values = []
    for start in progress(range(0, len(rows), 512), leave=False):
        ix = rows[start:start + 512]
        cols = np.broadcast_to(np.arange(k), (len(ix), k))
        ctx = choice_context_scores(data, ix, weights, False)
        x = choice_batch_matrix(data, ix, cols, ctx)
        values.append(model.predict(x, raw_score=True, num_threads=5).reshape(len(ix), k))
    return np.vstack(values).astype(np.float32)

def choice_get_names():
    return (choice_features_RICH / 'features.txt').read_text().splitlines() + (choice_features_V2 / 'names.txt').read_text().splitlines() + ['ctx_clicked', 'ctx_unseen', 'ctx_displayed', 'ctx_click_lift', 'ctx_margin']
base_choice_matrix = choice_batch_matrix

def choice_batch_matrix(dataset, rows, cols, ctx):
    base = base_choice_matrix(dataset, rows, cols, ctx)
    extra = campaign_features(dataset.name)[rows[:, None], cols].reshape(len(base), -1)
    return np.concatenate([base, extra], axis=1)
choice_TUNE = dict(params=dict(num_threads=4, histogram_pool_size=256))
raw_scores = {'test': {}, 'observed': {}}
choice_features_RICH = BASE_RICH
train, aux, exposure = [choice_Data(name) for name in ('train', 'aux', 'exposure')]
test, observed = (choice_Data('test'), choice_Data('observed'))
train_rows = np.arange(len(train.meta))
observed_rows = np.flatnonzero(observed.meta.end.to_numpy() >= START)
queries = {'test': (test, np.arange(len(test.meta))), 'observed': (observed, observed_rows)}
for seed in progress(SEEDS):
    model_SEED = seed
    model, context = choice_train_choice(train, aux, exposure, train_rows, CLASSES, choice_get_names() + CAMPAIGN_NAMES)
    for name, (data, rows) in queries.items():
        raw_scores[name].setdefault('choice', []).append(choice_score(model, context, data, rows, len(CLASSES)))
    del model, context
del train, aux, exposure, test, observed, queries
gc.collect()
finish_stage()


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/131 [00:00<?, ?it/s]

  0%|          | 0/59 [00:00<?, ?it/s]

  0%|          | 0/460 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

## 9. Обучение остальных бустингов: incumbent, specialist, routes


In [ ]:
class routes_RouteData(choice_Data):

    def __init__(self, name):
        super().__init__(name)
        self.rules, self.look, self.clock = [np.load(choice_features_V2 / f'{name}_route{k}.npy', mmap_mode='r') for k in range(3)]

def routes_matrix(data, rows, cols, classes):
    timing = np.broadcast_to(data.clock[rows, None, :], (*cols.shape, data.clock.shape[-1]))
    voted = data.rules[rows, None, :] == classes[cols, None]
    rich = data.rich[rows[:, None], cols]
    if routes_TUNE.get('zero'):
        rich = np.array(rich, copy=True)
        rich[..., routes_TUNE['zero']] = 0
    return np.concatenate([rich, voted, data.look[rows[:, None], cols], timing], axis=-1)
routes_TUNE = {}

def routes_fit_model(train, aux, rows, classes):
    names = (choice_features_RICH / 'features.txt').read_text().splitlines()
    ar = np.flatnonzero(aux.meta.client.isin(train.meta.client.iloc[rows]))
    sources = [(train, rows, 1.0), (aux, ar, routes_TUNE.get('aux_mult', 0.75))]
    ref = train.meta.end.iloc[rows].max()
    n = sum((len(i) for _, i, _ in sources))
    k = routes_TUNE.get('k', 9)
    dim = len(names) + train.look.shape[2] + train.rules.shape[1] + train.clock.shape[1]
    x = np.empty((n * k, dim), np.float16)
    y = np.empty((n, k), bool)
    w = np.empty(n, np.float32)
    offset = 0
    for data, idx, mult in sources:
        rng = np.random.default_rng(model_SEED + 109 + (data.name == 'aux') + 2 * (data.name == 'observed'))
        for i in progress(range(0, len(idx), 512), leave=False):
            ix = idx[i:i + 512]
            gold = np.searchsorted(classes, data.y[ix])
            u = rng.random((len(ix), len(classes)), dtype=np.float32)
            u[np.arange(len(ix)), gold] = -1
            cols = np.sort(np.argpartition(u, k - 1, axis=1)[:, :k], axis=1)
            end = offset + len(ix)
            x[offset * k:end * k] = routes_matrix(data, ix, cols, classes).reshape(-1, dim)
            y[offset:end] = cols == gold[:, None]
            w[offset:end] = mult * np.exp2(-(ref - data.meta.end.to_numpy()[ix]) / features_DAY / routes_TUNE.get('half_life', 28)) * regime_mult(data.meta.end.to_numpy()[ix])
            offset = end
    params = dict(objective='binary', num_leaves=63, learning_rate=0.075, min_data_in_leaf=110, max_bin=87, lambda_l2=8, feature_fraction=0.9, deterministic=True, seed=model_SEED, num_threads=5, force_col_wise=True, verbosity=-1)
    params.update(routes_TUNE.get('params', {}))
    cat = [i for i, n in enumerate(names) if n in ('last_product', 'last_banner', 'last_target', 'product') or n.startswith('end_event_')]
    cat.extend(range(dim - 6, dim - 2))
    ds = lgb.Dataset(x, label=y.ravel(), weight=np.repeat(w, k), categorical_feature=cat, params=params)
    ds.construct()
    del x
    return train_booster_progress(params, ds, num_boost_round=routes_TUNE.get('rounds', 530))

def routes_score(m, data, rows, classes):
    scores = []
    for i in progress(range(0, len(rows), 512), leave=False):
        ix = rows[i:i + 512]
        cols = np.broadcast_to(np.arange(len(classes)), (len(ix), len(classes)))
        x = routes_matrix(data, ix, cols, classes)
        scores.append(m.predict(x.reshape(-1, x.shape[-1]), raw_score=True, num_threads=5).reshape(len(ix), -1))
    return np.vstack(scores).astype(np.float32)
specialist_TUNE = {}

def specialist_select_context(sources, classes):
    n_features = sources[0][0].ctx.shape[1]
    sums = np.zeros((len(classes), n_features))
    squares = np.zeros(n_features)
    totals = np.zeros(len(classes))
    for data, rows, w in sources:
        for i in progress(range(0, len(rows), 512), leave=False):
            ix = rows[i:i + 512]
            x = data.ctx[ix].astype(np.float32)
            y = np.searchsorted(classes, data.y[ix])
            squares += np.square(x).sum(0) * w
            np.add.at(sums, y, x * w)
            totals += np.bincount(y, minlength=len(classes)) * w
    mean = sums.sum(0) / totals.sum()
    sd = np.maximum(0.09, np.sqrt(np.maximum(0.0001, squares / totals.sum() - mean ** 2)))
    delta = np.abs(sums / np.maximum(1, totals[:, None]) - mean) / sd
    groups = [(0, 160, 20), (160, 320, 14), (320, 480, 10), (480, 760, 36), (760, 1240, 15)]
    return [np.r_[np.concatenate([lo + np.argsort(-d[lo:hi], kind='stable')[:k] for lo, hi, k in groups]), 1240:1244] for d in delta]

def specialist_matrix(data, rows, col, fields):
    return np.hstack([data.rich[rows, col], data.extra[rows, col], data.ctx[rows[:, None], fields]])

def week_cv_run_incumbent(train, rows, queries, classes):
    names = (choice_features_RICH / 'features.txt').read_text().splitlines()
    out = {k: 0.0 for k in queries}
    for variant, width, coef in [('deep', names.index('market_view_1'), 0.35), ('long', len(names), 0.65)]:
        m = model_fit(train.rich[:, :, :width], rows, train.y, train.meta, classes, names[:width], variant, choice_features_RICH)
        for key, (data, qrows) in queries.items():
            out[key] = out[key] + coef * model_predict(m, data.rich[:, :, :width], qrows)
        del m
    return out

def week_cv_run_specialist(train, rows, queries, classes):
    aux = choice_Data('aux')
    end = train.meta.end.to_numpy()
    ar = np.flatnonzero(aux.meta.client.isin(train.meta.client.iloc[rows]))
    tune = specialist_TUNE
    sources = [(train, rows, 1.0), (aux, ar, tune.get('aux_mult', 0.45))]
    features = specialist_select_context(sources, classes)
    out = {k: np.empty((len(q), len(classes)), np.float32) for k, (_, q) in queries.items()}
    ref = end[rows].max()
    names = (choice_features_RICH / 'features.txt').read_text().splitlines()
    cat = [i for i, n in enumerate(names) if n.startswith('end_event_') or n in ('last_product', 'last_banner', 'last_target')]
    for col, (product, fields) in progress(enumerate(zip(classes, features)), total=len(classes), leave=False):
        rng = np.random.default_rng(model_SEED + int(product))
        positive = sum(((d.y[r] == product).sum() for d, r, w in sources))
        if positive < 60:
            for k in out:
                out[k][:, col] = float(np.log((positive + 0.5) / (len(rows) + len(ar) + 1)))
            continue
        negative = sum((len(r) for _, r, _ in sources)) - positive
        rate = min(1.0, (2500 + 11 * positive) / negative)
        xs, ys, ws = ([], [], [])
        for data, idx, weight in sources:
            gold = data.y[idx] == product
            keep = gold | (rng.random(len(idx)) < rate)
            sel = idx[keep]
            xs.append(specialist_matrix(data, sel, col, fields))
            ys.append(gold[keep])
            ws.append(weight * np.exp2(-(ref - data.meta.end.to_numpy()[sel]) / features_DAY / tune.get('half_life', 25)) * regime_mult(data.meta.end.to_numpy()[sel]))
        ds = lgb.Dataset(np.vstack(xs), label=np.concatenate(ys), weight=np.concatenate(ws), categorical_feature=cat)
        params = dict(objective='binary', verbosity=-1, learning_rate=0.065, num_leaves=23, max_bin=63, min_data_in_leaf=65, lambda_l2=10.0, feature_fraction=0.8, seed=model_SEED, num_threads=4, deterministic=True, force_col_wise=True)
        params.update(tune.get('params', {}))
        m = train_booster_progress(params, ds, num_boost_round=int(round(tune.get('rounds_mult', 1.0) * min(230, 80 + positive // 20))))
        for k, (data, qrows) in queries.items():
            for i in range(0, len(qrows), 1024):
                ix = qrows[i:i + 1024]
                out[k][i:i + len(ix), col] = m.predict(specialist_matrix(data, ix, col, fields), raw_score=True, num_threads=4) + np.log(rate)
    return out
choice_features_RICH = CAMP_RICH
specialist_TUNE = dict(params=dict(num_leaves=47, feature_fraction=0.6, min_data_in_leaf=200), half_life=50)
for seed in progress(SEEDS):
    model_SEED = seed
    train = routes_RouteData('train')
    aux = routes_RouteData('aux')
    test, observed = (routes_RouteData('test'), routes_RouteData('observed'))
    rows = np.arange(len(train.meta))
    queries = {'test': (test, np.arange(len(test.meta))), 'observed': (observed, observed_rows)}
    for name, run in progress([('incumbent', week_cv_run_incumbent), ('specialist', week_cv_run_specialist)], leave=False):
        scores = run(train, rows, queries, CLASSES)
        for dataset, values in scores.items():
            raw_scores[dataset].setdefault(name, []).append(values)
    model = routes_fit_model(train, aux, rows, CLASSES)
    for dataset, (data, selected) in queries.items():
        raw_scores[dataset].setdefault('routes', []).append(routes_score(model, data, selected, CLASSES))
    del model, train, aux, test, observed, queries
    gc.collect()
finish_stage()


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/540 [00:00<?, ?it/s]

... (35 outputs left out)

  0%|          | 0/88 [00:00<?, ?it/s]

  0%|          | 0/530 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

## 10. Три конфигурации трансформера


In [ ]:
seqx_train_DAY = 86400000
seqx_train_SEQ = ('ev', 'pr', 'dcut', 'dprv', 'hr')
seqx_train_DEV = DEVICE

class seqx_train_SeqX(nn.Module):

    def __init__(self, class_codes, n_feat, L, d=128, layers=2, heads=4, drop=0.1, dp=48):
        super().__init__()
        self.L, self.d, self.dp = (L, d, dp)
        self.ev = nn.Embedding(1100, 64, padding_idx=0)
        self.pr = nn.Embedding(72, 32, padding_idx=0)
        self.hr = nn.Embedding(24, 8)
        self.num = nn.Linear(2, 16)
        self.inp = nn.Linear(64 + 32 + 8 + 16, d)
        self.pos = nn.Parameter(torch.zeros(L, d))
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, drop, batch_first=True, norm_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.cls = nn.Embedding(len(class_codes), 32)
        self.q = nn.Linear(32, dp)
        self.pp = nn.Linear(d, dp)
        self.match = nn.Parameter(torch.tensor(2.0))
        self.cand = nn.Sequential(nn.Linear(n_feat, 96), nn.GELU())
        self.head = nn.Sequential(nn.Linear(d + dp + 32 + 96 + 2, 192), nn.GELU(), nn.Dropout(drop), nn.Linear(192, 1))
        self.register_buffer('cls_pr', torch.tensor(np.asarray(class_codes) + 2, dtype=torch.long))

    def forward(self, ev, pr, dcut, dprv, hr, cand):
        b, k = (ev.shape[0], self.cls_pr.shape[0])
        pad = ev == 0
        x = torch.cat([self.ev(ev), self.pr(pr), self.hr(hr), self.num(torch.stack([dcut, dprv], -1))], -1)
        h = self.norm(self.enc(self.inp(x) + self.pos, src_key_padding_mask=pad))
        m = (~pad).float().unsqueeze(-1)
        u = (h * m).sum(1) / m.sum(1).clamp(min=1)
        hp = self.pp(h)
        att = torch.einsum('bld,kd->bkl', hp, self.q(self.cls.weight)) / self.dp ** 0.5
        match = (pr.unsqueeze(1) == self.cls_pr.view(1, -1, 1)).float()
        att = (att + self.match * match).masked_fill(pad.unsqueeze(1), -10000.0)
        ctx = torch.einsum('bkl,bld->bkd', att.softmax(-1), hp)
        rec = (match * torch.arange(self.L, device=ev.device).float()).amax(-1, keepdim=True) / self.L
        z = torch.cat([u.unsqueeze(1).expand(-1, k, -1), ctx, self.cls.weight.unsqueeze(0).expand(b, -1, -1), self.cand(cand), torch.log1p(match.sum(-1, keepdim=True)), rec], -1)
        return self.head(z).squeeze(-1)

class seqx_train_Source:
    L = 128

    def __init__(self, name):
        s = np.load(seqx_train_D / f'{name}_seq.npz')
        self.seq = {k: s[k] for k in seqx_train_SEQ}
        self.y, self.end = (s['y'], s['end'])
        self.cand = np.load(seqx_train_D / f'{name}_cand.npy', mmap_mode='r')

    def batch(self, rows):
        rows = np.sort(rows)
        t = [torch.from_numpy(self.seq[k][rows, -self.L:].astype(np.int64 if k in ('ev', 'pr', 'hr') else np.float32)) for k in seqx_train_SEQ]
        return (rows, t + [torch.from_numpy(self.cand[rows].astype(np.float32))])

def seqx_train_predict(model, src, rows, bs=128):
    model.eval()
    out = np.empty((len(rows), 32), np.float32)
    order = np.argsort(rows)
    with torch.no_grad():
        for i in progress(range(0, len(rows), bs), leave=False):
            o = order[i:i + bs]
            _, t = src.batch(rows[o])
            out[o] = model(*[x.to(seqx_train_DEV) for x in t]).cpu().numpy()
    return out

def seqx_train_train(sources, parts, class_codes, n_feat, L, epochs, seed, lr, bs, half_life, evaluate=None, log=print, d=96, layers=2):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    ref = max((sources[n].end[r].max() for n, r, _ in parts if len(r) and (not hasattr(sources[n], 'q'))))
    items = []
    for j, (n, r, mult) in enumerate(parts):
        r = r[sources[n].y[r] >= 0]
        w = mult * np.exp2(-(ref - sources[n].end[r]) / seqx_train_DAY / half_life)
        items.append((np.full(len(r), j), r, w))
    pj, pr_, pw = (np.concatenate(x) for x in zip(*items))
    pw = (pw / pw.mean()).astype(np.float32)
    model = seqx_train_SeqX(class_codes, n_feat, L, d=d, layers=layers).to(seqx_train_DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(pj) / bs))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    t0 = time.time()
    for ep in progress(range(1, epochs + 1), leave=False):
        model.train()
        perm = rng.permutation(len(pj))
        tot = 0.0
        batch_progress = progress(range(0, len(perm), bs), leave=False)
        for i in batch_progress:
            sel = perm[i:i + bs]
            xs, ys, ws, qs = ([], [], [], [])
            for j in np.unique(pj[sel]):
                s = sel[pj[sel] == j]
                src = sources[parts[j][0]]
                rows, t = src.batch(pr_[s])
                ord_ = np.argsort(pr_[s])
                xs.append(t)
                ys.append(src.y[rows])
                ws.append(pw[s][ord_])
                qs.append(src.q[rows] if hasattr(src, 'q') else None)
            t = [torch.cat([x[k] for x in xs]).to(seqx_train_DEV) for k in range(len(xs[0]))]
            y = torch.from_numpy(np.concatenate(ys).astype(np.int64)).to(seqx_train_DEV)
            w = torch.from_numpy(np.concatenate(ws)).to(seqx_train_DEV)
            if all((q is None for q in qs)):
                loss = (F.cross_entropy(model(*t), y, reduction='none') * w).sum() / w.sum()
            else:
                q = torch.from_numpy(np.concatenate([np.eye(32, dtype=np.float32)[yy] if qq is None else qq for yy, qq in zip(ys, qs)])).to(seqx_train_DEV)
                loss = (-(q * F.log_softmax(model(*t), 1)).sum(1) * w).sum() / w.sum()
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            tot += loss.item() * len(sel)
        if evaluate:
            evaluate(ep, model)
    return model

class CompetitiveEncoder(nn.Module):

    def __init__(self, original_encoder, width):
        super().__init__()
        self.base = original_encoder
        self.relative = nn.Linear(3 * width, 96)
        nn.init.zeros_(self.relative.weight)
        nn.init.zeros_(self.relative.bias)

    def forward(self, candidates):
        centered = candidates - candidates.mean(1, keepdim=True)
        gap = candidates - candidates.amax(1, keepdim=True)
        scaled = centered / (candidates.std(1, keepdim=True, unbiased=False) + 0.2)
        contrasts = torch.cat([centered, gap, scaled.clamp(-5, 5)], dim=-1)
        return self.base(candidates) + 0.5 * torch.tanh(self.relative(contrasts))
OriginalSeqX = seqx_train_SeqX

class CompetitiveSeqX(OriginalSeqX):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        width = self.cand[0].in_features
        self.cand = CompetitiveEncoder(self.cand, width)
prepare_transformer_data()
seqx_train_Source.L = 128
sources = {name: seqx_train_Source(name) for name in ('train', 'aux')}
parts = [('train', np.arange(len(sources['train'].y)), 1.0), ('aux', np.arange(len(sources['aux'].y)), 0.55)]
n_features = sources['train'].cand.shape[2]
transformer_progress = progress(total=3)
for tag, lr, seeds, threads in [('seqxc', 0.001, [SEED], 3), ('seqxclr2', 0.002, [SEED], 3), ('competitive', 0.002, [SEED], 3)]:
    torch.set_num_threads(threads)
    seqx_train_SeqX = CompetitiveSeqX if tag == 'competitive' else OriginalSeqX
    for seed in seeds:
        CURRENT_TRAINING = f'{tag}, seed {seed}'
        model = seqx_train_train(sources, parts, CLASSES, n_features, 128, 4, seed, lr, TRAIN_BATCH_SIZE, 28, log=lambda text: None, d=96, layers=2)
        for dataset, rows in [('test', None), ('observed', observed_rows)]:
            query = seqx_train_Source(dataset)
            selected = np.arange(len(query.y)) if rows is None else rows
            raw_scores[dataset].setdefault(tag, []).append(seqx_train_predict(model, query, selected))
        del model
        transformer_progress.update(1)
transformer_progress.close()
finish_stage()


  0%|          | 0/100000 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/12359 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/45026 [00:00<?, ?it/s]

... (22 outputs left out)

  0%|          | 0/1134 [00:00<?, ?it/s]

  0%|          | 0/1134 [00:00<?, ?it/s]

  0%|          | 0/97 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

## 11. Объединение прогнозов


In [ ]:
def pool_predict(z, v):
    m = z.shape[1]
    p = softmax(z / np.exp(v[m:])[None, :, None], axis=2)
    return np.log(np.maximum(np.einsum('nmk,m->nk', p, softmax(v[:m])), 1e-12)).astype(np.float32)
RECIPE = {'weights': [0.00011954266532758235, 0.2106386623791732, 0.17540716530809078, 0.10070886478072846, 0.15677410186616716, 0.35635166300051285], 'temperatures': [1.0001745624779512, 0.9668534790844843, 0.9723829475449998, 1.0115880469828664, 1.0589605572642051, 1.1759607543068349], 'message': 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL'}
original_weight = RECIPE['weights'][-1]
original_temperature = RECIPE['temperatures'][-1]
RECIPE['weights'] = RECIPE['weights'][:-1] + [original_weight / 2, original_weight / 2]
RECIPE['temperatures'] = RECIPE['temperatures'] + [original_temperature]
experts = ['incumbent', 'choice', 'specialist', 'routes', 'seqxc', 'seqxclr2', 'competitive']
parameters = np.r_[np.log(RECIPE['weights']), np.log(RECIPE['temperatures'])]
pooled = {}
for dataset in ('test', 'observed'):
    scores = np.stack([np.mean(raw_scores[dataset][expert], axis=0) for expert in experts], axis=1)
    pooled[dataset] = pool_predict(scores.astype(np.float64), parameters).astype(np.float64)
finish_stage()


## 12. Калибровка по видимым кликам других групп клиентов


In [ ]:
rolling_calibration_BIN = 6 * 3600000

def rolling_calibration_features(data, rows, base):
    names = (choice_features_RICH / 'features.txt').read_text().splitlines()
    raw = data.rich[rows].astype(np.float32)

    def get(name):
        return raw[:, :, names.index(name)]
    values = [np.clip(base - base.max(1, keepdims=True), -8, 0) / 4, get('views_last_15') / 3, -np.minimum(get('view_events'), 8) / 4, -np.minimum(get('target_events'), 12) / 6, get('market_click_12') / 4, get('market_view_12') / 4, get('target_clicks') / 2]
    x = np.stack(values, axis=-1)
    x -= x.mean(1, keepdims=True)
    return x.clip(-3, 3).astype(np.float32)

def calib_variants_fit(base, x, y, weights, penalty, bound):
    n, k, d = x.shape

    def objective(coef):
        bias, theta = (coef[:k], coef[k:])
        logits = base + bias[None, :] + np.einsum('nkd,d->nk', x, theta)
        loss = np.dot(weights, logsumexp(logits, axis=1) - logits[np.arange(n), y])
        error = softmax(logits, axis=1)
        error[np.arange(n), y] -= 1
        error *= weights[:, None]
        grad = np.r_[error.sum(0), np.einsum('nk,nkd->d', error, x)]
        return (float(loss + 0.5 * penalty * np.square(coef).sum()), grad + penalty * coef)
    r = minimize(objective, np.zeros(k + d), method='L-BFGS-B', jac=True, bounds=[(-bound, bound)] * k + [(-0.5, 0.5)] * d, options=dict(maxiter=70, gtol=1e-06, ftol=1e-10, maxcor=8))
    return r.x

def calib_variants_calibrate(meta, obase, ox, y_all, qcuts, qbase, qx, group, half_life, penalty, bound, cw=None):
    out = qbase.copy()
    sel = np.flatnonzero(meta.allowed_class.to_numpy() & (meta.client_fold.to_numpy() != group))
    ct, clients = (meta.click_time.to_numpy()[sel], meta.client.to_numpy()[sel])
    u, inv, cnt = np.unique(clients, return_inverse=True, return_counts=True)
    for cut in progress(np.unique(qcuts), leave=False):
        take = np.flatnonzero(qcuts == cut)
        w = np.exp2(-np.abs(cut - ct) / features_DAY / half_life) / cnt[inv]
        if cw is not None:
            w = w * cw[y_all[sel]]
        coef = calib_variants_fit(obase[sel], ox[sel], y_all[sel], w, penalty, bound)
        out[take] += coef[:32][None] + np.einsum('nkd,d->nk', qx[take], coef[32:])
    return out
test, observed = (choice_Data('test'), choice_Data('observed'))
meta = observed.meta.iloc[observed_rows]
y_observed = np.searchsorted(CLASSES, observed.y[observed_rows])
observed_features = rolling_calibration_features(observed, observed_rows, pooled['observed'])
test_features = rolling_calibration_features(test, np.arange(len(test.meta)), pooled['test'])
cuts = test.meta.end.to_numpy() // rolling_calibration_BIN * rolling_calibration_BIN
scores = pooled['test'].copy()
for group in progress(range(5)):
    rows = np.flatnonzero(test.meta.client.to_numpy() % 5 == group)
    scores[rows] = calib_variants_calibrate(meta, pooled['observed'], observed_features, y_observed, cuts[rows], pooled['test'][rows], test_features[rows], group, 3.0, 5.0, 1.0)
finish_stage()


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/53 [00:00<?, ?it/s]

  0%|          | 0/54 [00:00<?, ?it/s]

  0%|          | 0/53 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/52 [00:00<?, ?it/s]

## 13. Сабмит


In [ ]:
log_probabilities = log_softmax(scores, axis=1)
prediction = pd.Series(CLASSES[log_probabilities.argmax(1)], index=test.meta.example_id)
submission = sample_submission.copy()
submission['encoded_product'] = submission.example_id.map(prediction).astype(int)
output = OUTPUT_DIR / 'submission.csv'
submission.to_csv(output, index=False, lineterminator='\n')
np.save(OUTPUT_DIR / 'submission_scores.npy', log_probabilities)
print(output)
finish_stage()
pipeline_progress.close()


/content/submission.csv
